# Find and union fold

find: initially set node -> node in dict, then set node -> parent

Union Fold: using rank to determine who is the parent, then low_rank -> high_rank. If the same rank, WLOG, the rank of one will promote.


Imagine start from two leaves, the algorithm ends when two leaves meet the same ancestor.

In [ ]:

parent = list(range(6))  # 6 people: 0..5
rank   = [0] * 6

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb: return
    if rank[ra] < rank[rb]: parent[ra] = rb
    elif rank[ra] > rank[rb]: parent[rb] = ra
    else: parent[rb] = ra; rank[ra] += 1

In [ ]:
def sum(a,b):
    return 5
def sum(a,b):
    return 7
print(sum(a=2,b=3))

7


[1,2], set {1->1,2->2}, since 1 != 2: {1->2,2->2}

[1,3], set {1->2,2->2,3->3}.In find(1), 1 != 2, so parent[1] = find(2) = 2, pu,pv = 2,3; and parent[2] = 3;

[2,3], set is {1->2,2->3,3->3}. p2 = find(2),since 2 != 3, parent[2] = find(3) = 3



In [ ]:

def findRedundantConnection(edges):
    parent = {}

    def find(x):
        if x not in parent:
            parent[x] = x
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    
    def find(x): ## same but shorter using setdefault
        if x != parent.setdefault(x, x): # put 1 into parent, handle the longer chain ???
            parent[x] = find(parent[x])  # Path compression
            print(f"parent[{x}] updated to {parent[x]} after path compression")
        return parent[x]
    
    for u, v in edges:
        print()
        print(f"Processing edge: {u}, {v}")
        print(f"Current parent mapping: {parent}")
        pu, pv = find(u), find(v)
        print(f"Find results: pu={pu}, pv={pv}")
        if pu == pv:
            return [u, v]  # according to find, the node have the same ancestor, can't connect
        parent[pu] = pv  # Union the sets
    
    return []  # No redundant connection found

print(findRedundantConnection([[1,2], [1,3], [2,3]]))


Processing edge: 1, 2
Current parent mapping: {}
Find results: pu=1, pv=2

Processing edge: 1, 3
Current parent mapping: {1: 2, 2: 2}
parent[1] updated to 2 after path compression
Find results: pu=2, pv=3

Processing edge: 2, 3
Current parent mapping: {1: 2, 2: 3, 3: 3}
parent[2] updated to 3 after path compression
Find results: pu=3, pv=3
[2, 3]


In [ ]:
class Solution(object):
    def accountsMerge(self, accounts: list[list[str]]):
        parent = {}
        rank = {}

        def find(x):
            # Lazy initialization: first time seeing this email,
            # it becomes its own parent (its own group leader)
            if x not in parent:
                parent[x] = x
                rank[x] = 0
            # Iterative path compression with path splitting:
            # each node skips one level, flattening the tree over time
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(x, y):
            rx, ry = find(x), find(y)
            if rx == ry:
                return  # Already in the same group
            # Union by rank: attach shorter tree under taller tree
            if rank[rx] < rank[ry]:
                parent[rx] = ry
            elif rank[rx] > rank[ry]:
                parent[ry] = rx
            else:
                parent[ry] = rx
                rank[rx] += 1

        # We need to remember which name goes with which email
        email_to_name = {}

        # Phase 1: For each account, union ALL its emails together
        # by chaining them to the first email in the list
        for account in accounts:
            name = account[0]
            first_email = account[1]
            for email in account[1:]:
                email_to_name[email] = name
                union(first_email, email)

        # Phase 2: Group all emails by their root representative
        from collections import defaultdict
        groups = defaultdict(list)
        for email in email_to_name:
            root = find(email)
            groups[root].append(email)

        # Phase 3: Build result — name + sorted emails for each group
        return [
            [email_to_name[root]] + sorted(emails)
            for root, emails in groups.items()
        ]

# 323.Number of Connected Components in an Undirected Graph

You have a graph of n nodes. You are given an integer n and an array edges where edges[i] = [ai, bi] indicates that there is an edge between ai and bi in the graph.

Return the number of connected components in the graph.

In [ ]:
class Solution(object):
    def countComponents(self, n, edges):
        """
        :type n: int
        :type edges: List[List[int]]
        :rtype: int
        """
        parent = list(range(n))
        rank = [0] * n

        def find(x):
            if parent[x] != x:# not point to yourself
                parent[x] = find(parent[x])
            return parent[x]

        def union(a,b):
            ra,rb = find(a), find(b)
            if ra == rb: return 0
            if rank[ra] < rank[rb]:parent[ra] = rb
            elif rank[ra] > rank[rb]: parent[rb] = ra
            else:  parent[rb] = ra;ra += 1
            return 1

        components = n
        for a,b in edges:
            components -= union(a,b)
        return components

[0, 1, 2, 3]
